# 03 Market-Making Backtester Skeleton

This notebook shows a generic inventory-aware market-making simulator on synthetic data.

The objective is to make the mechanics visible: fair value, quotes, fills, inventory, spread capture, mark-to-market, costs, and inventory penalty. The quote logic is deliberately simple and not a live strategy.

## Step 0: Imports And Paths

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

from market_making.inventory import update_inventory, inventory_penalty
from market_making.quote_simulator import make_inventory_skewed_quote

## Step 1: Create A Synthetic Fair-Value Table

The fair value and external signal are generated from toy random processes.

In [ ]:
rng = np.random.default_rng(42)

timestamps = pd.date_range(
    start="2026-01-05 14:30:00+00:00",
    periods=300,
    freq="1min",
)

fair_value = 0.50 + np.cumsum(rng.normal(0.0, 0.003, size=len(timestamps)))
fair_value = np.clip(fair_value, 0.05, 0.95)
external_signal = pd.Series(rng.normal(0.0, 1.0, size=len(timestamps))).rolling(15, min_periods=1).mean()

mm_table = pd.DataFrame({
    "timestamp_utc": timestamps,
    "fair_value": fair_value,
    "external_signal": external_signal,
})

mm_table["fair_value_change"] = mm_table["fair_value"].diff().fillna(0.0)

display(mm_table.head())

## Step 2: Quote Parameters

These are toy parameters for a public simulator. They are not calibrated to any real market.

In [ ]:
BASE_SPREAD = 0.04
INVENTORY_SKEW = 0.006
FILL_SIZE = 1.0
FILL_PROBABILITY = 0.18
TRANSACTION_COST = 0.001
INVENTORY_PENALTY_RATE = 0.002
MAX_ABS_INVENTORY = 8.0

params = {
    "base_spread": BASE_SPREAD,
    "inventory_skew": INVENTORY_SKEW,
    "fill_size": FILL_SIZE,
    "fill_probability": FILL_PROBABILITY,
    "transaction_cost": TRANSACTION_COST,
    "inventory_penalty_rate": INVENTORY_PENALTY_RATE,
    "max_abs_inventory": MAX_ABS_INVENTORY,
}

display(pd.DataFrame([params]))

## Step 3: Walk Forward Through Quotes And Fills

The loop records every quote. Fills are synthetic and depend on toy random draws. Inventory limits prevent the simulator from accumulating unlimited exposure.

In [ ]:
quote_rows = []
fill_rows = []

inventory = 0.0
cash = 0.0
spread_capture = 0.0
cost_paid = 0.0

for idx, row in mm_table.iterrows():
    quote = make_inventory_skewed_quote(
        fair_value=row["fair_value"],
        base_spread=BASE_SPREAD,
        inventory=inventory,
        inventory_skew=INVENTORY_SKEW,
    )

    can_buy = inventory + FILL_SIZE <= MAX_ABS_INVENTORY
    can_sell = inventory - FILL_SIZE >= -MAX_ABS_INVENTORY

    bid_fill = can_buy and (rng.random() < FILL_PROBABILITY)
    ask_fill = can_sell and (rng.random() < FILL_PROBABILITY)

    if bid_fill:
        inventory = update_inventory(inventory, FILL_SIZE, side="buy")
        cash -= quote["bid"] * FILL_SIZE
        spread_capture += max(row["fair_value"] - quote["bid"], 0.0) * FILL_SIZE
        cost_paid += TRANSACTION_COST * FILL_SIZE
        fill_rows.append({
            "timestamp_utc": row["timestamp_utc"],
            "side": "buy",
            "fill_price": quote["bid"],
            "fair_value": row["fair_value"],
            "inventory_after": inventory,
        })

    if ask_fill:
        inventory = update_inventory(inventory, FILL_SIZE, side="sell")
        cash += quote["ask"] * FILL_SIZE
        spread_capture += max(quote["ask"] - row["fair_value"], 0.0) * FILL_SIZE
        cost_paid += TRANSACTION_COST * FILL_SIZE
        fill_rows.append({
            "timestamp_utc": row["timestamp_utc"],
            "side": "sell",
            "fill_price": quote["ask"],
            "fair_value": row["fair_value"],
            "inventory_after": inventory,
        })

    mark_to_market = cash + inventory * row["fair_value"]
    inv_penalty = inventory_penalty(
        inventory,
        penalty_rate=INVENTORY_PENALTY_RATE,
    )

    quote_rows.append({
        "timestamp_utc": row["timestamp_utc"],
        "fair_value": row["fair_value"],
        "external_signal": row["external_signal"],
        "bid": quote["bid"],
        "ask": quote["ask"],
        "inventory": inventory,
        "cash": cash,
        "spread_capture": spread_capture,
        "cost_paid": cost_paid,
        "inventory_penalty": inv_penalty,
        "net_liquidation_value": mark_to_market - cost_paid - inv_penalty,
    })

quote_log = pd.DataFrame(quote_rows)
fill_log = pd.DataFrame(fill_rows)

display(quote_log.head())
display(fill_log.head())

## Step 4: Simulator Summary

The summary separates trading mechanics from inventory pressure.

In [ ]:
summary = {
    "quote_rows": len(quote_log),
    "fills": len(fill_log),
    "final_inventory": quote_log["inventory"].iloc[-1],
    "max_abs_inventory": quote_log["inventory"].abs().max(),
    "spread_capture": quote_log["spread_capture"].iloc[-1],
    "cost_paid": quote_log["cost_paid"].iloc[-1],
    "ending_net_liquidation_value": quote_log["net_liquidation_value"].iloc[-1],
}

summary_df = pd.DataFrame([summary])
display(summary_df)

if len(fill_log) > 0:
    fill_summary = (
        fill_log
        .groupby("side", as_index=False)
        .agg(
            fills=("side", "size"),
            avg_fill_price=("fill_price", "mean"),
            avg_inventory_after=("inventory_after", "mean"),
        )
    )
    display(fill_summary)

## Step 5: Visual Check

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

axes[0].plot(quote_log["timestamp_utc"], quote_log["fair_value"], label="fair value")
axes[0].plot(quote_log["timestamp_utc"], quote_log["bid"], alpha=0.7, label="bid")
axes[0].plot(quote_log["timestamp_utc"], quote_log["ask"], alpha=0.7, label="ask")
axes[0].legend()
axes[0].set_title("Synthetic fair value and quotes")

axes[1].step(quote_log["timestamp_utc"], quote_log["inventory"], where="post")
axes[1].set_title("Inventory")

axes[2].plot(quote_log["timestamp_utc"], quote_log["spread_capture"], label="spread capture")
axes[2].plot(quote_log["timestamp_utc"], quote_log["cost_paid"], label="cost paid")
axes[2].legend()
axes[2].set_title("Spread capture and costs")

axes[3].plot(quote_log["timestamp_utc"], quote_log["net_liquidation_value"])
axes[3].set_title("Toy net liquidation value")

for ax in axes:
    ax.grid(True, alpha=0.25)

fig.tight_layout()
plt.show()

## Public Version Omissions

This simulator does not include real order-book data, queue position, live fill modelling, private fair-value estimates, proprietary quote adjustment rules, or production risk controls. It is a skeleton for explaining the mechanics of inventory-aware market making.